# GW Spectrum from Decaying Turbulence -- Self-Similar Derivation

**Branch:** `decaying-selfsimilar`   **Baseline for cross-checks:** `decaying_turbulence_gw_spectrum_analytic.ipynb` (quasi-stationary, BK2016 mode-by-mode).

---

## Why a new derivation

The earlier derivation applied the BK2016 temporal ansatz $f(\tau;k) = (1+\tau/\tau_1(k))^{-2/3}$ mode-by-mode on top of a time-independent Kolmogorov spectrum. That silently assumes the two-point correlator factorises as

$$F_{ij}(\mathbf{k},\tau_1,\tau_2) = A(k)\,P_{ij}(\hat k)\,f(\tau_1-\tau_2;\,k),$$

i.e. $k$ and $t$ separate. **This is wrong for decaying turbulence.** Self-similar decay gives a spectrum whose *shape* evolves in time:

$$E(k,t) = A(t)\,F\!\bigl(k/k_p(t)\bigr),$$

with an amplitude $A(t)$ and a peak wavenumber $k_p(t)$ that both move. $k$ and $t$ are entangled through the dimensionless argument $k/k_p(t)$ -- there is no separable $P(k)\cdot f(\tau;k)$.

The goal of this notebook is to redo the derivation keeping this entanglement from the start, and to recover the quasi-stationary result as the short-time limit $\Delta t \ll \tau_{\rm decay}$.

### References (to cite as derivation proceeds)

- **G2007** Gogoberidze, Kahniashvili & Kosowsky, PRD **76**, 123504 (2007) -- general kernel $H_{ijij}(k,\omega)$.
- **K2008** Kahniashvili et al., PRD **78**, 123006 (2008) -- aeroacoustic $k_{\rm GW}\to 0$ reduction.
- **BK2016** Brandenburg & Kahniashvili, PRL **114**, 075001 (2015) -- self-similar decay laws, $\phi$ scaling function.
- **Loitsiansky / Saffman invariants** -- fix the $A(t)$, $k_p(t)$ power laws depending on which invariant is conserved.

---


## Structure

1. Self-similar ansatz: $E(k,t)$, $A(t)$, $k_p(t)$, scaling function $\phi$.
2. Two-time, two-point correlator -- where separability actually fails and what replaces $f(\tau;k)$.
3. Temporal Fourier transform in the self-similar frame.
4. GW production kernel $H_{ijij}(\mathbf{k},\omega)$ with the non-factorised correlator.
5. Aeroacoustic $k_{\rm GW}\to 0$ reduction -- which parts of K2008 survive, which do not.
6. Dimensional counting and the wavenumber integral.
7. Main result -- to be boxed when derivation closes.
8. Quasi-stationary limit: recover the BK2016 baseline as $\Delta t\ll\tau_{\rm decay}$. **Required sanity check.**
9. Connection to `core.py` -- what changes (or: what new function is needed).
10. Numerical cross-check against baseline notebook.

Cells below are scaffolding. Fill in derivations inline; keep `TODO` markers where a choice has to be made (invariant, ansatz form, closure).


In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import matplotlib.pyplot as plt
import mpmath as mp
from scipy import integrate, special

# Compatibility shim: np.trapezoid was added in numpy 2.0; older numpy only has np.trapz.
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

# Baseline numerics to compare against (quasi-stationary BK2016):
# from gw_turbulence.core import g_decaying, _temporal_conv_decay, H_pq_decaying


In [2]:
# =====================================================================
# Notebook-wide plotting style.  Edit THIS cell to restyle every figure.
# =====================================================================

# Wong 2011 colorblind-safe palette, brightness-tuned for screen + print.
PALETTE = [
    "#0072B2",  # blue
    "#D55E00",  # vermillion
    "#009E73",  # bluish green
    "#CC79A7",  # reddish purple
    "#E69F00",  # orange
    "#56B4E9",  # sky blue
    "#F0E442",  # yellow
    "#000000",  # black
]

plt.rcParams.update({
    "figure.figsize":      (8.0, 5.0),
    "figure.dpi":          110,
    "savefig.dpi":         150,
    "font.size":           15,
    "axes.titlesize":      17,
    "axes.labelsize":      16,
    "xtick.labelsize":     13,
    "ytick.labelsize":     13,
    "legend.fontsize":     13,
    "lines.linewidth":     2.6,
    "axes.linewidth":      1.4,
    "xtick.major.width":   1.4,
    "ytick.major.width":   1.4,
    "xtick.major.size":    6,
    "ytick.major.size":    6,
    "xtick.minor.visible": False,
    "ytick.minor.visible": False,
    "axes.grid":           False,
    "axes.prop_cycle":     plt.cycler(color=PALETTE),
    "legend.frameon":      False,
    "axes.spines.top":     False,
    "axes.spines.right":   False,
})

def tidy(ax, nx=5, ny=5):
    """Cap an axes at roughly (nx, ny) major ticks. No-ops on log axes (they self-place)."""
    from matplotlib.ticker import MaxNLocator
    if ax.get_xscale() == "linear":
        ax.xaxis.set_major_locator(MaxNLocator(nbins=nx))
    if ax.get_yscale() == "linear":
        ax.yaxis.set_major_locator(MaxNLocator(nbins=ny))
    return ax


## 1. Self-similar ansatz

In decaying isotropic turbulence the energy spectrum collapses onto a single self-similar form (Batchelor; BK2016):

$$E(k,t) \;=\; u^2(t)\,L(t)\,\phi\!\bigl(\kappa\bigr),
\qquad \kappa \equiv k\,L(t),\qquad L(t) = 1/k_p(t),$$

with $u^2(t)$ the (twice-)mean kinetic energy and $L(t)$ the integral length. The scaling function $\phi(\kappa)$ has a fixed IR rise $\phi\sim\kappa^{s}$ and a Kolmogorov inertial tail $\phi\sim\kappa^{-5/3}$. For non-helical hydro we adopt the **Loitsiansky** invariant ($s=4$) used by BK2016 as the default class.

### 1.1 Power laws from invariant + closure

Two conditions fix $u(t)$ and $L(t)$:

**(a) Invariant in the IR.**  As $k\to 0$, $\phi(\kappa)\sim C_\phi\kappa^s$, so
$$E(k,t)\Big|_{k\to 0} \;=\; u^2(t)\,L^{s+1}(t)\,C_\phi\,k^{s}.$$
Loitsiansky ($s=4$) requires the coefficient of $k^4$ to be a constant of motion:
$$u^2(t)\,L^5(t) \;=\; \text{const}.$$

**(b) Self-similar decay closure.**  Dimensionally, $du^2/dt \sim -u^3/L$, i.e. $L/u\sim t$.

Writing $u^2\sim t^{-p}$ and $L\sim t^{q}$, conditions (a) and (b) give

$$-p+5q=0,\qquad q+p/2=1 \;\Rightarrow\; \boxed{p=10/7,\;\; q=2/7}\quad(\text{Loitsiansky}).$$

For comparison, Saffman ($s=2$) gives $p=6/5,\,q=2/5$. We carry $(p,q)$ symbolically so the same code reproduces either class.

### 1.2 Shifted-time form

For a primordial source switched on at $t=0$ with onset time $\tau_\ast$, the regular ansatz is

$$u^2(t) \;=\; u_0^2\,(1+t/\tau_\ast)^{-p},
\qquad L(t) \;=\; L_0\,(1+t/\tau_\ast)^{q},
\qquad k_p(t)=1/L(t).$$

The instantaneous dissipation rate follows from $\varepsilon(t) = -\tfrac{1}{2}\,d u^2/dt$:

$$\varepsilon(t) \;=\; \frac{p\,u_0^2}{2\,\tau_\ast}\,(1+t/\tau_\ast)^{-p-1}.$$

This is the only place a *physical* timescale enters: at $t\ll\tau_\ast$ the turbulence is "fresh"; at $t\gg\tau_\ast$ it is in self-similar decay with $u^2\propto t^{-p}$.

### 1.3 Local eddy time

The Kolmogorov/BK2016 eddy turnover time at wavenumber $k$ and time $t$ is

$$\tau_1(k,t) \;=\; \bigl[\varepsilon(t)^{1/3}\,k^{2/3}\bigr]^{-1}.$$

In the inertial range and at fixed $k$, $\tau_1(k,t)$ grows with $t$ (the turbulence becomes "slower" as it decays). Two ratios will control everything downstream:

- $\omega\,\tau_1(k,t)$ — the dimensionless frequency at scale $k$ and time $t$.
- $\tau_1(k,t)/\tau_\ast$ — local eddy time vs. decay time; **small in the inertial range** and *the* expansion parameter for §3.


In [3]:
# Self-similar amplitude, integral scale, peak wavenumber, dissipation, spectrum.
# Default class: Loitsiansky (p=10/7, q=2/7, IR slope s=4).
# Pass p, q, s in `pars` to switch class (Saffman: p=6/5, q=2/5, s=2).

def u_t(t, **pars):
    """u(t) = u_0 (1 + t/tau_*)^{-p/2}, so u^2 ~ t^{-p}."""
    u0, tau_st, p = pars["u0"], pars["tau_st"], pars["p"]
    return u0 * (1 + t/tau_st) ** (-p/2)

def L_t(t, **pars):
    """L(t) = L_0 (1 + t/tau_*)^{q}.  Exponent is q (not q/2) — L ~ t^q is the BK2016 law."""
    l0, tau_st, q = pars["l0"], pars["tau_st"], pars["q"]
    return l0 * (1 + t/tau_st) ** q

def k_p_t(t, **pars):
    return 1.0 / L_t(t, **pars)

def eps_t(t, **pars):
    """eps(t) = -(1/2) d u^2/dt = (p u_0^2 / 2 tau_*) (1 + t/tau_*)^{-p-1}."""
    u0, tau_st, p = pars["u0"], pars["tau_st"], pars["p"]
    return p * u0**2 / (2*tau_st) * (1 + t/tau_st) ** (-p - 1)

def tau1(k, t, **pars):
    """Local eddy turnover time at wavenumber k and time t."""
    return 1.0 / (eps_t(t, **pars)**(1/3) * k**(2/3))

def phi(kappa, s=4):
    """Self-similar spectrum shape: IR ~ kappa^s, UV ~ kappa^{-5/3}."""
    return kappa**s / (1 + kappa) ** (s + 5/3)

def E_kt(k, t, **pars):
    """Energy spectrum E(k,t) = u^2(t) L(t) phi(k L(t))."""
    L = L_t(t, **pars)
    return u_t(t, **pars)**2 * L * phi(k*L, s=pars.get("s", 4))


## 2. Two-time, two-point correlator

### 2.1 Spectral tensor and equal-time normalization

For incompressible, isotropic, statistically homogeneous turbulence, the velocity correlator in Fourier space is transverse and rotationally symmetric:

$$\bigl\langle u_i(\mathbf{k},t_1)\,u_j^*(\mathbf{k}',t_2)\bigr\rangle
\;=\; (2\pi)^3\,\delta^3(\mathbf{k}-\mathbf{k}')\,P_{ij}(\hat{k})\,\Phi(k;t_1,t_2),
\qquad P_{ij}(\hat k) = \delta_{ij}-\hat k_i\hat k_j.$$

The (one-sided) equal-time energy spectrum $E(k,t)$ is the orientation-summed spherical shell density. Tracing and integrating over $\mathbf{k}$ fixes the normalization

$$\Phi(k;t,t) \;=\; \frac{E(k,t)}{4\pi k^2}.$$

So $\Phi$ inherits all the spectral-tensor convention; everything physical we know about scales lives in $E(k,t)$ from §1.

### 2.2 Self-similar two-time ansatz

For decaying (non-stationary) turbulence we cannot write $\Phi(k;t_1,t_2)=A(k)\,f(t_1-t_2;k)$. The symmetric, IR-correct generalisation is the **geometric-mean amplitude with a normalised decorrelation**:

$$\boxed{\;\Phi(k;t_1,t_2) \;=\; \sqrt{\Phi(k,t_1)\,\Phi(k,t_2)}\;\mathcal{R}(k;t_1,t_2)\;}
\qquad\mathcal{R}(k;t,t)=1.$$

This object:

- is symmetric in $t_1\leftrightarrow t_2$ (positivity-friendly under Cauchy–Schwarz once $|\mathcal{R}|\le 1$);
- reduces to the stationary $A(k)\,f(t_1-t_2;k)$ whenever $\Phi(k,t)\to A(k)$ and $\mathcal{R}\to f(t_1-t_2;k)$;
- and is **not** factorisable in $k,t$: through $\Phi(k,t_i) \propto u^2(t_i)\,L(t_i)\,\phi(kL(t_i))$, both the amplitude and the shape evolve with $t_i$.

### 2.3 Decorrelation — BK2016 evaluated at the midpoint

Following BK2016 we take a local-eddy-turnover decorrelation, but with the eddy time $\tau_1$ evaluated at the **midpoint** $T\equiv(t_1+t_2)/2$ so that the kernel is symmetric in $t_1\leftrightarrow t_2$:

$$\mathcal{R}(k;t_1,t_2) \;=\; \left(\,1+\frac{|t_1-t_2|}{\tau_1\!\bigl(k,T\bigr)}\,\right)^{-2/3},
\qquad \tau_1(k,T)=\bigl[\varepsilon(T)^{1/3}k^{2/3}\bigr]^{-1}.$$

Two limits worth checking right now:

1. **Quasi-stationary**: $T$ frozen, $\tau_1(k,T)\to\tau_1(k)$, $\Phi(k,t)\to A(k)$. Then $\mathcal{R}\to(1+|\tau|/\tau_1(k))^{-2/3}$, the BK2016 form.
2. **Wide separation** $|t_1-t_2|\to\infty$: $\mathcal{R}\to 0$, so memory of the source is lost.

Other defensible choices for $\mathcal{R}$ (Kraichnan-sweeping exponential, or Gaussian in $\tau/\tau_1$) would change inertial-range tails but not the order-of-magnitude conclusions of §3. We carry the BK2016 form by default; the cell below isolates `R` so swapping in another kernel is a one-line change.


In [4]:
# Two-time velocity correlator Phi(k; t1, t2) under the self-similar ansatz.
#   amplitude:   geometric mean of equal-time spectra
#   decorrelation: BK2016 power law with tau_1 evaluated at midpoint T = (t1+t2)/2

def R_decorr(k, t1, t2, **pars):
    """Decorrelation factor R(k; t1, t2). Reduces to BK2016 f(tau;k) in the stationary limit."""
    T = 0.5 * (t1 + t2)
    return (1.0 + np.abs(t1 - t2) / tau1(k, T, **pars)) ** (-2/3)

def Phi_eq(k, t, **pars):
    """Equal-time spectral tensor density: Phi(k,t) = E(k,t) / (4 pi k^2)."""
    return E_kt(k, t, **pars) / (4 * np.pi * k**2)

def Phi(k, t1, t2, **pars):
    """Two-time spectral density Phi(k; t1, t2) = sqrt(Phi(k,t1) Phi(k,t2)) * R(k; t1, t2)."""
    return np.sqrt(Phi_eq(k, t1, **pars) * Phi_eq(k, t2, **pars)) * R_decorr(k, t1, t2, **pars)


## 3. Wigner–Ville transform of the two-time correlator

### 3.1 Why not a single $\omega$

In stationary turbulence $\Phi(k;t_1,t_2)=\Phi_{\rm st}(k;t_1-t_2)$, so a single Fourier transform in $\tau=t_1-t_2$ gives a function of $\omega$ alone. In the decaying ansatz of §2, $\Phi$ depends separately on $T=(t_1+t_2)/2$ and $\tau=t_1-t_2$, so the natural spectral object is the **Wigner–Ville transform**

$$\boxed{\;W_\Phi(k;\omega,T) \;\equiv\; \int_{-\infty}^{\infty} d\tau\;e^{i\omega\tau}\;\Phi\!\bigl(k;\,T+\tfrac{\tau}{2},\,T-\tfrac{\tau}{2}\bigr).\;}$$

It is a function of three arguments. In the stationary limit it loses its $T$-dependence and collapses back to the usual single-frequency power spectrum. The GW source in §4 will be expressed in terms of $W_\Phi$.

### 3.2 Slow/fast expansion

Split the two-time correlator into a slow envelope and a fast decorrelation:

$$\Phi\!\bigl(k;\,T+\tfrac{\tau}{2},\,T-\tfrac{\tau}{2}\bigr)
\;=\; \underbrace{\sqrt{\Phi(k,T+\tau/2)\,\Phi(k,T-\tau/2)}}_{\text{slow: varies on }\tau_\ast}
\,\;\underbrace{\bigl(1+|\tau|/\tau_1(k,T)\bigr)^{-2/3}}_{\text{fast: decays on }\tau_1(k,T)}.$$

Set $f(t)\equiv\ln\Phi(k,t)$ (treating $k$ as a parameter). By even symmetry,

$$f(T+\tau/2)+f(T-\tau/2) \;=\; 2f(T) + \tfrac{\tau^2}{4}\,f''(T) + \tfrac{\tau^4}{192}\,f^{(4)}(T)+\cdots$$

so the slow envelope is

$$\sqrt{\Phi(k,T+\tau/2)\,\Phi(k,T-\tau/2)}
\;=\; \Phi(k,T)\,\Bigl[\,1+\tfrac{\tau^2}{8}\,\partial_T^2\ln\Phi(k,T) + O(\tau^4)\,\Bigr].$$

The Wigner integral is dominated by $|\tau|\lesssim \tau_1(k,T)$. With $|\partial_T^2\ln\Phi|\sim 1/(\tau_\ast+T)^2$, the expansion parameter is

$$\boxed{\;\epsilon \;\equiv\; \frac{\tau_1(k,T)}{\tau_\ast+T} \;\ll\;1\quad\text{(inertial range, } kL(T)\gg1\text{).}}$$

### 3.3 Leading order — instantaneous BK2016

Dropping the $O(\epsilon^2)$ slow correction, with $\sigma\equiv\tau/\tau_1(k,T)$,

$$\begin{aligned}
W_\Phi^{(0)}(k;\omega,T)
&= \Phi(k,T)\int_{-\infty}^{\infty}\!d\tau\;e^{i\omega\tau}\bigl(1+|\tau|/\tau_1(k,T)\bigr)^{-2/3}\\
&= \Phi(k,T)\,\tau_1(k,T)\,\hat g\!\bigl(\omega\,\tau_1(k,T)\bigr),
\end{aligned}$$

with

$$\hat g(q) \;=\; \int_{-\infty}^{\infty}\!d\sigma\;e^{iq\sigma}\bigl(1+|\sigma|\bigr)^{-2/3}
\;=\; 2\,\mathrm{Re}\!\int_0^{\infty}\!d\sigma\;e^{iq\sigma}\bigl(1+\sigma\bigr)^{-2/3}.$$

This is **identical** to the BK2016 temporal kernel `g_decaying` in `src/gw_turbulence/core.py`. The only difference from the baseline notebook is that every parameter on the right is evaluated at the slow time $T$:

$$\boxed{\;W_\Phi^{(0)}(k;\omega,T) \;=\; \frac{E(k,T)}{4\pi k^2}\;\tau_1(k,T)\;\hat g\!\bigl(\omega\,\tau_1(k,T)\bigr).\;}$$

**Conceptual upshot.** The quasi-stationary BK2016 expression is the *leading order in $\epsilon=\tau_1/\tau_{\rm decay}$* of the self-similar derivation, taken **at each instant $T$**. The non-separability of §2 is real, but at leading order it shows up only by making $\Phi$, $\tau_1$ functions of $T$ inside an otherwise stationary-looking formula. The genuinely new physics enters via two routes:

1. **Slow $T$-integration of the GW source** (§7) — the spectrum $E(k,T)$ and eddy time $\tau_1(k,T)$ drift over the emission window.
2. **Sub-leading correction $W_\Phi^{(1)}$** below — memory of decay *within* a single eddy turnover.

### 3.4 First sub-leading correction

Putting the slow expansion back:

$$W_\Phi^{(1)}(k;\omega,T)
\;=\; \Phi(k,T)\,\tau_1(k,T)\;\frac{\tau_1^2(k,T)}{8}\,\partial_T^2\ln\Phi(k,T)\;\hat g_2\!\bigl(\omega\tau_1(k,T)\bigr),$$

with

$$\hat g_2(q) \;\equiv\; \int_{-\infty}^{\infty}\!d\sigma\;\sigma^2\,e^{iq\sigma}\bigl(1+|\sigma|\bigr)^{-2/3} \;=\; -\partial_q^2\hat g(q).$$

So the relative correction to the leading-order kernel is

$$\frac{W_\Phi^{(1)}}{W_\Phi^{(0)}} \;=\; \frac{\tau_1^2(k,T)}{8}\,\partial_T^2\ln\Phi(k,T)\;\frac{\hat g_2(q)}{\hat g(q)}
\;\sim\;\epsilon^2,$$

an explicit, computable $O(\epsilon^2)$ piece. The cell below codes both $W_\Phi^{(0)}$ and the full Wigner integral, so the size of the correction can be checked numerically before propagating into §4.


In [5]:
# Wigner-Ville transform of Phi: leading-order closed form vs full numerical FT.

def g_hat(q, sigma_max=200.0, n=4001):
    """Dimensionless BK2016 temporal kernel:
       g_hat(q) = 2 Re int_0^inf d sigma exp(i q sigma) (1+sigma)^{-2/3}.
    Matches `g_decaying` in src/gw_turbulence/core.py up to convention."""
    sig = np.linspace(0.0, sigma_max, n)
    integrand = np.exp(1j * q * sig) * (1.0 + sig) ** (-2/3)
    return 2.0 * np.trapezoid(integrand, sig).real

def W_phi_leading(k, omega, T, **pars):
    """W_Phi^{(0)}(k; omega, T) = Phi(k,T) * tau_1(k,T) * g_hat(omega tau_1(k,T))."""
    t1 = tau1(k, T, **pars)
    return Phi_eq(k, T, **pars) * t1 * g_hat(omega * t1)

def W_phi_full(k, omega, T, tau_span=None, n_tau=2001, **pars):
    """Numerical Wigner integral of the full self-similar Phi(k; T+tau/2, T-tau/2).

    The shifted-time ansatz u^2 ~ (1+t/tau_*)^{-p} is singular at t = -tau_*, so
    tau is clamped to keep both endpoints in (-tau_*, inf). For T too close to 0,
    the available window can be narrower than the decorrelation kernel; in that
    regime the leading-order analytic result W_phi_leading is the right answer
    anyway.
    """
    t1 = tau1(k, T, **pars)
    tau_st = pars["tau_st"]
    span_natural = 30.0 * t1
    span_safe    = 1.8 * (T + tau_st)       # keep T +/- tau/2 strictly inside (-tau_*, inf)
    span = min(tau_span if tau_span is not None else span_natural, span_safe)
    taus = np.linspace(-span, span, n_tau)
    vals = np.array([Phi(k, T + tau/2, T - tau/2, **pars) for tau in taus])
    return np.trapezoid(vals * np.exp(1j * omega * taus), taus).real

def epsilon_param(k, T, **pars):
    """Expansion parameter epsilon = tau_1(k,T) / (tau_* + T)."""
    return tau1(k, T, **pars) / (pars["tau_st"] + T)


---

## V. Verification of the §1–3 modeling choices

Before propagating §1–3 into the G2007/K2008 reduction, four assumptions deserve explicit scrutiny because each one feeds into the $O(\epsilon^2)$ correction that we want to carry through:

- **(V.a)** the self-similar closure $L/u\sim t$ underlying $(p,q)$,
- **(V.b)** the geometric-mean amplitude ansatz $\Phi(k;t_1,t_2)=\sqrt{\Phi_1\Phi_2}\,\mathcal{R}$,
- **(V.c)** the midpoint convention $\tau_1(k,T)$ with $T=(t_1+t_2)/2$,
- **(V.d)** the BK2016 power-law decorrelation vs. the Kraichnan exponential.

### V.a — Closure $L/u\sim t$

The closure is a consequence, not an extra assumption, once we accept self-similar collapse plus the standard dimensional dissipation estimate.

**Energy budget.** $\dfrac{d}{dt}\!\left(\tfrac{1}{2}u^2\right) = -\varepsilon$.

**Dimensional dissipation.** Self-similar collapse $E(k,t)=u^2 L\,\phi(kL)$ means the only integral-scale quantities are $u(t)$, $L(t)$. Dimensionally, $\varepsilon$ must be built from these:
$$\varepsilon(t) \;=\; C_\varepsilon\,\frac{u^3(t)}{L(t)},$$
with a dimensionless constant $C_\varepsilon$ set by $\phi$. *No other timescale exists* in the self-similar regime — that is what "self-similar" buys you.

**Substitute and read off.** With $u^2=u_0^2(1+t/\tau_\ast)^{-p}$ and $L=L_0(1+t/\tau_\ast)^q$, both sides scale as $(1+t/\tau_\ast)^a$ for some $a$:

- LHS exponent: $-p-1$.
- RHS exponent: $-3p/2 - q$.

Matching: $-p-1 = -3p/2-q$, i.e.
$$\boxed{\,q+\tfrac{p}{2}=1\,.}$$
Combined with the invariant condition from §1.1, this fixes $(p,q)$. The closure is therefore the *content* of self-similarity plus dimensional dissipation — not a separate modelling step. Any modification would require a second timescale, which would break self-similar collapse at the level of $E(k,t)$ itself.

**Where it could fail.** Only outside the self-similar window: very early ($t\ll\tau_\ast$, "still injecting"), very late (anomalous late-time scaling from boundary effects in a finite box), or when a second mechanism (helicity, magnetic field, compressibility) carries its own timescale. None of these apply to the non-helical hydro problem we're solving. **(V.a) accepted.**


### V.b — Geometric-mean amplitude

The ansatz of §2.2,
$$\Phi(k;t_1,t_2) \;=\; \sqrt{\Phi(k,t_1)\,\Phi(k,t_2)}\;\mathcal{R}(k;t_1,t_2),$$
is **not an approximation**; it is the canonical Cauchy–Schwarz parameterization of any two-time correlator. For each fixed $k$, the family $\{u_i(\mathbf{k},t)\}_t$ are random variables with finite second moment $\Phi(k,t,t)=\Phi(k,t)$. Cauchy–Schwarz gives

$$|\Phi(k;t_1,t_2)|\;\le\;\sqrt{\Phi(k,t_1)\,\Phi(k,t_2)}\;\;\;\Longleftrightarrow\;\;\;|\mathcal{R}(k;t_1,t_2)|\le 1,$$

so we lose nothing by writing $\Phi=\sqrt{\Phi_1\Phi_2}\,\mathcal{R}$ and putting all model content in a *bounded* correlation coefficient $\mathcal{R}$. (Stationary limit: $\mathcal{R}\to f(t_1-t_2;k)$, the BK2016 kernel.)

**Why the alternatives are worse.**

- *Arithmetic-mean amplitude*  $\Phi \stackrel{?}{=}\dfrac{\Phi_1+\Phi_2}{2}\,\mathcal{R}'$ requires $|\mathcal{R}'|\le 2\sqrt{\Phi_1\Phi_2}/(\Phi_1+\Phi_2)$ — a bound that depends on $\Phi_1,\Phi_2$ themselves and is $<1$ whenever $\Phi_1\neq\Phi_2$. So $\mathcal{R}'$ is no longer a clean correlation coefficient.
- *Earlier-time-frozen amplitude*  $\Phi \stackrel{?}{=}\Phi(k,\min(t_1,t_2))\,\mathcal{R}''$ requires $|\mathcal{R}''|\le \sqrt{\Phi_{\max}/\Phi_{\min}}$, which is unbounded as the source decays — also pathological as a correlation coefficient.

The cell below illustrates the second issue numerically: it picks $\Phi_1\ne\Phi_2$, plugs in the *same* physically meaningful BK2016 decorrelation, and checks the implied $\mathcal{R}'$ (arithmetic mean) and $\mathcal{R}''$ (earlier-time) against the unit bound. The geometric-mean $\mathcal{R}$ stays inside $[-1,1]$ by construction.


In [6]:
# V.b -- check that geometric-mean R is bounded, while arithmetic-mean R' and
# earlier-time R'' can exceed the unit bound when Phi_1 != Phi_2.

# We hold the *physical* Phi(k; t1, t2) fixed at the BK2016 form and just relabel
# how we split it between "amplitude" and "correlation coefficient":
#   geom:    Phi = sqrt(Phi_1 Phi_2) * R_geom        ->  R_geom  must satisfy |R| <= 1
#   arith:   Phi = (Phi_1 + Phi_2)/2 * R_arith       ->  bound depends on Phi_1, Phi_2
#   earlier: Phi = Phi(min(t_1, t_2)) * R_earlier    ->  bound depends on Phi_max/Phi_min

pars_v = dict(u0=1.0, l0=1.0, tau_st=1.0, p=10/7, q=2/7, s=4)

def _phi_geom_target(k, t1, t2, **pars):
    """Physical Phi(k; t1, t2) used as the reference -- BK2016 decorrelation."""
    return np.sqrt(Phi_eq(k, t1, **pars) * Phi_eq(k, t2, **pars)) * R_decorr(k, t1, t2, **pars)

print(f"{'k':>5} {'t1':>6} {'t2':>6} {'Phi1/Phi2':>10} {'|R_geom|':>10} {'|R_arith|':>10} {'|R_earlier|':>12}")
for k in [1.0, 10.0]:
    # Widely separated times so Phi_1, Phi_2 differ noticeably
    for t1, t2 in [(0.2, 5.0), (0.5, 10.0), (1.0, 20.0)]:
        Phi1 = Phi_eq(k, t1, **pars_v); Phi2 = Phi_eq(k, t2, **pars_v)
        Phi12 = _phi_geom_target(k, t1, t2, **pars_v)
        R_geom    = Phi12 / np.sqrt(Phi1 * Phi2)
        R_arith   = Phi12 / ((Phi1 + Phi2) / 2)
        R_earlier = Phi12 / min(Phi1, Phi2)
        print(f"{k:5.1f} {t1:6.2f} {t2:6.2f} {Phi1/Phi2:10.3f} {abs(R_geom):10.4f} {abs(R_arith):10.4f} {abs(R_earlier):12.4f}")

print()
print("R_geom stays <= 1 (by Cauchy-Schwarz).")
print("R_arith and R_earlier can exceed 1 once Phi_1 != Phi_2: they are no longer correlation coefficients.")


    k     t1     t2  Phi1/Phi2   |R_geom|  |R_arith|  |R_earlier|
  1.0   0.20   5.00      4.413     0.5398     0.4190       1.1341
  1.0   0.50  10.00      6.887     0.4888     0.3253       1.2827
  1.0   1.00  20.00     10.975     0.4465     0.2470       1.4792
 10.0   0.20   5.00     11.265     0.2487     0.1361       0.8349
 10.0   0.50  10.00     20.513     0.2163     0.0911       0.9796
 10.0   1.00  20.00     36.336     0.1917     0.0619       1.1555

R_geom stays <= 1 (by Cauchy-Schwarz).
R_arith and R_earlier can exceed 1 once Phi_1 != Phi_2: they are no longer correlation coefficients.


### V.c — Midpoint vs. symmetric-average vs. earlier-time evaluation of $\tau_1$

The decorrelation kernel uses a time-dependent eddy time $\tau_1(k,t)$. We have to specify *at what time* to evaluate it inside $\mathcal{R}(k;t_1,t_2)$. Three candidates:

| Choice | $\tau_1$ inside $\mathcal{R}$ | Symmetric in $t_1\leftrightarrow t_2$? |
|---|---|---|
| Midpoint (used in §2) | $\tau_1\!\bigl(k,\,(t_1+t_2)/2\bigr)$ | yes |
| Symmetric average | $\tfrac{1}{2}\bigl[\tau_1(k,t_1)+\tau_1(k,t_2)\bigr]$ | yes |
| Earlier time | $\tau_1\!\bigl(k,\,\min(t_1,t_2)\bigr)$ | **no** |

**Earlier-time is excluded.** $\Phi$ must be real and symmetric in $(t_1,t_2)$ (reality + isotropy), and $\mathcal{R}$ inherits both properties. Evaluating $\tau_1$ at $\min(t_1,t_2)$ gives a $\mathcal{R}$ with a discontinuous derivative at $t_1=t_2$ and breaks reflection invariance — it shows up as an $O(\epsilon)$ (not $\epsilon^2$) deviation from the symmetric choices.

**Midpoint vs. symmetric-average agree to $O(\epsilon^4)$.** Set $T=(t_1+t_2)/2$, $\Delta=(t_1-t_2)/2$, and Taylor-expand:

$$\tfrac{1}{2}\bigl[\tau_1(k,T+\Delta)+\tau_1(k,T-\Delta)\bigr]
\;=\; \tau_1(k,T) + \tfrac{\Delta^2}{2}\,\partial_T^2\tau_1(k,T) + O(\Delta^4).$$

So the *fractional* difference between midpoint and symmetric-average is

$$\frac{\delta\tau_1}{\tau_1} \;=\; \frac{\Delta^2}{2}\,\frac{\partial_T^2\tau_1}{\tau_1}
\;\sim\;\left(\frac{\Delta}{\tau_\ast+T}\right)^2.$$

Inside the Wigner integral $\Delta\sim\tau_1$, hence $\delta\tau_1/\tau_1\sim\epsilon^2$. Because $\mathcal{R}$ depends on $\tau_1$ through the dimensionless combination $|\tau|/\tau_1$ and the Wigner integral has typical support $|\tau|\sim\tau_1$, a fractional change $\delta\tau_1/\tau_1=O(\epsilon^2)$ changes $W_\Phi$ by $O(\epsilon^2)$ in the *prefactor*. **But this contribution sits inside the $W_\Phi^{(1)}$ term we are already computing in §3.4** — it's not a separate effect, just a $C_\varepsilon$-style choice of $O(1)$ coefficient inside that correction.

**Conclusion.** Midpoint is the simplest symmetric choice and is what we keep. The symmetric-average gives the same leading-order $W_\Phi^{(0)}$ and the same $O(\epsilon^2)$ *parametric scaling*; only the dimensionless coefficient in $W_\Phi^{(1)}$ shifts. The cell below shows the size of that shift numerically — it's well within the $O(\epsilon^2)$ tolerance.


In [7]:
# V.c -- compare midpoint vs. symmetric-average tau_1 evaluation in the Wigner integral.
# Earlier-time choice excluded analytically (breaks t1<->t2 symmetry).

def R_decorr_symavg(k, t1, t2, **pars):
    """Decorrelation with tau_1 evaluated as the symmetric average of the two endpoints."""
    tau1_sym = 0.5 * (tau1(k, t1, **pars) + tau1(k, t2, **pars))
    return (1.0 + np.abs(t1 - t2) / tau1_sym) ** (-2/3)

def Phi_symavg(k, t1, t2, **pars):
    return np.sqrt(Phi_eq(k, t1, **pars) * Phi_eq(k, t2, **pars)) * R_decorr_symavg(k, t1, t2, **pars)

def W_phi_full_symavg(k, omega, T, n_tau=2001, **pars):
    t1 = tau1(k, T, **pars)
    span = min(30.0*t1, 1.8*(T + pars["tau_st"]))
    taus = np.linspace(-span, span, n_tau)
    vals = np.array([Phi_symavg(k, T+tau/2, T-tau/2, **pars) for tau in taus])
    return np.trapezoid(vals * np.exp(1j*omega*taus), taus).real

print(f"{'k':>6} {'T':>6} {'eps':>8} {'omega':>6} {'W_mid':>12} {'W_symavg':>12} {'diff/W':>10} {'eps^2':>8}")
for k in [10.0, 100.0, 1000.0]:
    for T in [3.0, 10.0]:
        eps = epsilon_param(k, T, **pars_v)
        for om in [0.5, 2.0]:
            Wm = W_phi_full(k, om, T, n_tau=4001, **pars_v)
            Ws = W_phi_full_symavg(k, om, T, n_tau=4001, **pars_v)
            rel = (Ws - Wm)/Wm if Wm else float('nan')
            print(f"{k:6.1f} {T:6.2f} {eps:8.4f} {om:6.2f} {Wm:12.4e} {Ws:12.4e} {rel:10.2e} {eps**2:8.4f}")
print()
print("Both choices give the same W to O(eps^2). The midpoint convention is kept.")


     k      T      eps  omega        W_mid     W_symavg     diff/W    eps^2
  10.0   3.00   0.1851   0.50  -1.2461e-07  -3.3097e-08  -7.34e-01   0.0343
  10.0   3.00   0.1851   2.00   9.6815e-07   9.3972e-07  -2.94e-02   0.0343
  10.0  10.00   0.1526   0.50  -7.4805e-08  -5.2785e-08  -2.94e-01   0.0233
  10.0  10.00   0.1526   2.00   1.8909e-07   1.8193e-07  -3.79e-02   0.0233
 100.0   3.00   0.0399   0.50   4.2228e-10   4.2290e-10   1.47e-03   0.0016
 100.0   3.00   0.0399   2.00   1.0325e-10   1.0367e-10   4.04e-03   0.0016
 100.0  10.00   0.0329   0.50   5.1574e-11   5.1830e-11   4.98e-03   0.0011
 100.0  10.00   0.0329   2.00   2.4691e-11   2.4669e-11  -9.08e-04   0.0011
1000.0   3.00   0.0086   0.50   3.6215e-14   3.6209e-14  -1.55e-04   0.0001
1000.0   3.00   0.0086   2.00   2.4103e-14   2.4103e-14  -2.15e-05   0.0001
1000.0  10.00   0.0071   0.50   1.4289e-14   1.4288e-14  -8.27e-05   0.0001
1000.0  10.00   0.0071   2.00   2.6468e-15   2.6478e-15   3.92e-04   0.0001

Both choice

### V.d — BK2016 power-law vs. Kraichnan exponential/Gaussian decorrelation

The model dependence on the *form* of $\mathcal{R}$ in $\tau$ is real and shows up most strongly at $\omega\tau_1\gg 1$ (high frequencies, short eddies), where heavy-tailed vs. light-tailed kernels differ dramatically.

| Kernel | Form (with $\sigma=\tau/\tau_1$) | High-$q$ Fourier tail of $\hat g(q)$ |
|---|---|---|
| BK2016 (sweeping, intermittent) | $(1+\|\sigma\|)^{-2/3}$ | power-law, $\sim q^{-2}$ from $\sigma=0$ corner |
| Kraichnan Gaussian (sweeping, mean-field) | $\exp(-\sigma^2/2)$ | Gaussian, $\sim e^{-q^2/2}$ |
| Lagrangian exponential | $\exp(-\|\sigma\|)$ | Lorentzian, $\sim 2/(1+q^2)$ |

**What survives the choice, what doesn't.**

- The **leading-order reduction** to a quasi-stationary-looking $W_\Phi^{(0)} = \Phi(k,T)\,\tau_1(k,T)\,\hat g(\omega\tau_1)$ survives any kernel of the form $\mathcal{R}=\mathcal{R}(|\tau|/\tau_1(k,T))$ — that's the universal structure of §3.
- The **$O(\epsilon^2)$ correction** $W_\Phi^{(1)}$ retains the same parametric size $\tau_1^2\partial_T^2\ln\Phi/8$, but its coefficient $\hat g_2/\hat g$ depends on the kernel.
- The **high-frequency spectrum** $\hat g(\omega\tau_1)$ at $\omega\tau_1\gg 1$ depends qualitatively on the choice: a power-law GW spectrum tail with BK2016, an exponentially cut-off tail with Kraichnan-Gaussian.

The baseline notebook (`decaying_turbulence_gw_spectrum_analytic.ipynb`) and `core.py` already use the BK2016 power law, so we carry that forward as the default. The cell below plots all three $\hat g(q)$ side-by-side so the high-$q$ behaviour is visible.


In [8]:
# V.d -- plot the three dimensionless temporal kernels side-by-side.

def g_hat_bk(q, sigma_max=200.0, n=8001):
    """BK2016: integrand (1+|sigma|)^{-2/3}."""
    sig = np.linspace(0.0, sigma_max, n)
    return 2.0 * np.trapezoid(np.exp(1j*q*sig) * (1.0 + sig)**(-2/3), sig).real

def g_hat_gauss(q, sigma_max=20.0, n=8001):
    """Kraichnan Gaussian: integrand exp(-sigma^2/2). Analytic value sqrt(2 pi) exp(-q^2/2)."""
    sig = np.linspace(0.0, sigma_max, n)
    return 2.0 * np.trapezoid(np.exp(1j*q*sig) * np.exp(-sig**2/2), sig).real

def g_hat_exp(q, sigma_max=50.0, n=8001):
    """Lagrangian exponential: integrand exp(-|sigma|). Analytic value 2/(1+q^2)."""
    sig = np.linspace(0.0, sigma_max, n)
    return 2.0 * np.trapezoid(np.exp(1j*q*sig) * np.exp(-sig), sig).real

qs = np.geomspace(0.05, 30, 80)
gB = np.array([g_hat_bk(q)    for q in qs])
gG = np.array([g_hat_gauss(q) for q in qs])
gE = np.array([g_hat_exp(q)   for q in qs])

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a in ax:
    a.plot(qs, np.abs(gB), label=r"BK2016 $(1+|\sigma|)^{-2/3}$")
    a.plot(qs, np.abs(gG), label=r"Kraichnan $\exp(-\sigma^2/2)$")
    a.plot(qs, np.abs(gE), label=r"Lagrangian $\exp(-|\sigma|)$")
    a.set_xlabel(r"$q = \omega\,\tau_1$")
    a.set_ylabel(r"$|\hat g(q)|$")
ax[0].set_title("linear")
tidy(ax[0])
ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_title("log-log")
ax[0].legend(loc="best")
plt.tight_layout()
plt.show()

print(f"q = 20:  BK = {abs(g_hat_bk(20)):.3e}   "
      f"Gauss = {abs(g_hat_gauss(20)):.3e}   "
      f"Exp = {abs(g_hat_exp(20)):.3e}    (analytic exp: 2/(1+20^2) = {2/(1+400):.3e})")
print("Heavy-tail BK and Lorentzian exp survive at q=20; Gaussian is below floating-point.")


q = 20:  BK = 1.418e-03   Gauss = 2.286e-17   Exp = 4.994e-03    (analytic exp: 2/(1+20^2) = 4.988e-03)
Heavy-tail BK and Lorentzian exp survive at q=20; Gaussian is below floating-point.


/tmp/ipykernel_42893/2122580578.py:36: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 4. GW production kernel with the non-factorised correlator

### 4.1 G2007 starting form

Under Gaussian closure for incompressible isotropic turbulence, the unequal-time correlator of the TT-projected anisotropic stress reduces to a convolution of velocity correlators (G2007):

$$\bigl\langle\Pi^{\rm TT}_{ij}(\mathbf{k},t_1)\,\Pi^{\rm TT}_{ij}{}^{\!*}(\mathbf{k},t_2)\bigr\rangle_{\!c}
\;=\; \int\!\frac{d^3k_1}{(2\pi)^3}\,\mathcal{K}(\mathbf{k},\mathbf{k}_1)\;
\Phi(k_1;t_1,t_2)\,\Phi(u;t_1,t_2),
\qquad \mathbf{u}=\mathbf{k}-\mathbf{k}_1.$$

The geometric kernel $\mathcal{K}$ is purely spatial — it comes from contracting two transverse projectors $P_{ij}(\hat{k}_i)$ with the TT projector. **It carries no $t_1,t_2$ dependence.** All time dependence sits in the two $\Phi$ factors.

For a decaying source the relevant spectral object is the **slow-time-resolved** Wigner transform of the stress correlator:

$$H_{ijij}(\mathbf{k},\omega;T) \;\equiv\; \int_{-\infty}^{\infty}\!d\tau\,e^{i\omega\tau}\,
\bigl\langle\Pi^{\rm TT}_{ij}(\mathbf{k},T+\tau/2)\,\Pi^{\rm TT}_{ij}{}^{\!*}(\mathbf{k},T-\tau/2)\bigr\rangle_{\!c}.$$

The total GW production then integrates over emission time:

$$H_{ijij}(\mathbf{k},\omega) \;=\; \int_0^{T_{\rm em}}\!dT\;H_{ijij}(\mathbf{k},\omega;T).$$

### 4.2 Wigner of a product = $\omega$-convolution of Wigners (at fixed $T$)

The crucial identity that lets §3 propagate into §4: for any two functions $g(t_1,t_2)$, $h(t_1,t_2)$, the Wigner transform in $\tau$ at fixed $T$ of their product factorises into a one-dimensional convolution in $\omega$:

$$W_{gh}(\omega,T) \;=\; \frac{1}{2\pi}\int d\omega'\;W_g(\omega',T)\,W_h(\omega-\omega',T).$$

*Proof.* $W_{gh}(\omega,T) = \int d\tau\,e^{i\omega\tau}g(T+\tau/2,T-\tau/2)\,h(T+\tau/2,T-\tau/2)$, which is the FT in $\tau$ at fixed $T$ of a product; by the standard FT convolution theorem this is $(2\pi)^{-1}$ times the convolution of $W_g$ and $W_h$ in $\omega$. ∎

Applying this to the G2007 reduction with $g=\Phi(k_1;\cdot,\cdot)$, $h=\Phi(u;\cdot,\cdot)$:

$$\boxed{\;H_{ijij}(\mathbf{k},\omega;T) \;=\; \int\!\frac{d^3k_1}{(2\pi)^3}\,\mathcal{K}(\mathbf{k},\mathbf{k}_1)\;\frac{1}{2\pi}\!\int\!d\omega_1\;W_\Phi(k_1;\omega_1,T)\,W_\Phi(u;\omega-\omega_1,T).\;}$$

This has **the same structure** as the stationary G2007 expression with $A(k)\tilde f(\omega,k)$ replaced by $W_\Phi(k;\omega,T)$. The geometric integral is untouched; the only change is that the temporal/spectral factor is now Wigner-Ville rather than a single power spectrum.

### 4.3 Carrying $O(\epsilon^2)$ through the convolution

From §3, $W_\Phi = W_\Phi^{(0)} + W_\Phi^{(1)} + O(\epsilon^4)$ where

$$W_\Phi^{(0)}(k;\omega,T) = \Phi(k,T)\,\tau_1(k,T)\,\hat g(\omega\tau_1(k,T)),
\qquad
W_\Phi^{(1)}(k;\omega,T) = \tfrac{\tau_1^2(k,T)}{8}\partial_T^2\ln\Phi(k,T)\;\Phi(k,T)\,\tau_1(k,T)\,\hat g_2(\omega\tau_1(k,T)),$$
with $\hat g_2(q)=-\hat g''(q)$. The convolution expands:

$$W_\Phi*W_\Phi \;=\; W_\Phi^{(0)}*W_\Phi^{(0)} \;+\; \bigl[\,W_\Phi^{(0)}*W_\Phi^{(1)}+W_\Phi^{(1)}*W_\Phi^{(0)}\,\bigr]\;+\;O(\epsilon^4),$$

and the bracketed cross terms are equal (convolution is commutative). Using $\hat g*\hat g \;=\;G_{\rm BK}$ and $\hat g*\hat g_2 = -G_{\rm BK}''$ (FT of $\sigma^2$ pulls down a $-\partial_q^2$):

- **Leading**: $(W_\Phi^{(0)}*W_\Phi^{(0)})(\omega) \;=\; \Phi(k_1,T)\Phi(u,T)\,\sqrt{\tau_1(k_1,T)\tau_1(u,T)}\,\tilde G_{\rm BK}^{(k_1,u)}(\omega;T)$
- **Cross** ($O(\epsilon^2)$): proportional to the same convolution but with one $\hat g$ replaced by $-\hat g''$, and a prefactor $\tfrac{\tau_1^2}{8}\partial_T^2\ln\Phi$ from whichever leg carried the $W_\Phi^{(1)}$.

For general $\mathbf{k}\neq 0$ the two legs have $k_1\neq u$ and the convolution doesn't reduce to a single-variable kernel. **The simplification happens in §5** when the aeroacoustic limit $\mathbf{k}\to 0$ forces $u\to k_1$, collapsing both legs to the same wavenumber and the same $\tau_1$.

The code cell below implements both the leading-order and the cross-term convolutions at fixed $(k_1,u)$. We use them in §5 with $u=k_1$.


In [9]:
# W_Phi * W_Phi convolution at u = k_1 (the K2008 case used in section 5).
# Carries both the leading G_BK and the cross-term -G_BK'' coming from W_Phi^(1).
#
# At u = k_1, both legs share the same tau_1 and Phi, and:
#   (W_Phi^(0) * W_Phi^(0))(omega) = Phi^2 * tau_1 * G_BK(omega tau_1)
#   2 (W_Phi^(0) * W_Phi^(1))(omega) = -(tau_1^3/4) * Phi^2 * d2lnPhi_dT2 * G_BK''(omega tau_1)
# so  W_PhiPhi(k_1; omega, T) = [Phi^2 * tau_1 / (2 pi)] * G_SS(q; xi),
# with q = omega tau_1, xi = tau_1^2 * d2 ln Phi / dT^2 (~ epsilon^2),
# and  G_SS(q; xi) = G_BK(q) - (xi/4) * G_BK''(q).

def G_BK(q, sigma_max=200.0, n=4001):
    """Auto-convolution G_BK(q) = (g_hat * g_hat)(q).  Matches _temporal_conv_decay in core.py."""
    sig = np.linspace(0.0, sigma_max, n)
    qs  = np.linspace(-3*max(abs(q), 1.0) - 5.0, 3*max(abs(q), 1.0) + 5.0, 2001)  # dense around q
    # Compute g_hat on a q-grid, then convolve numerically (cheap O(N^2) is fine for one-off).
    ghat = np.array([2.0 * np.trapezoid(np.exp(1j*qq*sig)*(1+sig)**(-2/3), sig).real for qq in qs])
    return float(np.trapezoid(np.interp(q - qs, qs, ghat, left=0, right=0) * ghat, qs))

def _gbk_curve(qs_grid, sigma_max=200.0, n=4001):
    """Evaluate g_hat on a uniform q-grid; used to derive G_BK and G_BK''."""
    sig = np.linspace(0.0, sigma_max, n)
    return np.array([2.0 * np.trapezoid(np.exp(1j*q*sig)*(1+sig)**(-2/3), sig).real for q in qs_grid])

def make_G_SS_table(q_max=30.0, n_q=601, sigma_max=200.0, n_sig=4001):
    """Pre-tabulate G_BK(q) and G_BK''(q) on a uniform grid for fast G_SS(q; xi)."""
    qs   = np.linspace(-q_max, q_max, n_q)
    g    = _gbk_curve(qs, sigma_max=sigma_max, n=n_sig)
    # Autoconvolution G_BK = g_hat * g_hat (trapezoid in q on the uniform grid).
    dq   = qs[1] - qs[0]
    GBK  = np.convolve(g, g, mode="same") * dq
    GBK2 = np.gradient(np.gradient(GBK, dq), dq)
    return qs, GBK, GBK2

_QS_GSS, _GBK_TAB, _GBK2_TAB = make_G_SS_table()

def G_SS(q, xi):
    """G_SS(q; xi) = G_BK(q) - (xi/4) G_BK''(q).  Tabulated; clipped to grid."""
    GBK  = np.interp(q, _QS_GSS, _GBK_TAB,  left=0.0, right=0.0)
    GBK2 = np.interp(q, _QS_GSS, _GBK2_TAB, left=0.0, right=0.0)
    return GBK - (xi / 4.0) * GBK2

# Sanity: G_SS(q, 0) should equal G_BK(q).
_q_test = np.array([0.5, 1.5, 5.0])
print("G_SS(q, 0) at q =", _q_test, "->", G_SS(_q_test, 0.0))
print("G_BK ref via numpy interp ->", np.interp(_q_test, _QS_GSS, _GBK_TAB))


G_SS(q, 0) at q = [0.5 1.5 5. ] -> [12.0784865   3.94299249  0.75806108]
G_BK ref via numpy interp -> [12.0784865   3.94299249  0.75806108]


## 5. Aeroacoustic limit $k_{\rm GW}\to 0$

For long-wavelength GW production ($k_{\rm GW}$ much smaller than the inverse integral scale), set $\mathbf{k}=0$ in §4. K2008 showed that in this limit the G2007 geometric kernel collapses to a constant:

$$\mathbf{k}\to 0\;\Rightarrow\;\mathbf{u}=-\mathbf{k}_1,\;\;u=k_1,\;\;\mathcal{K}(0,\mathbf{k}_1)=\frac{14}{3}.$$

This is purely a property of the angular integration of the TT projector — independent of which temporal/spectral model is on the two legs. So **the spatial reduction commutes with the §3–§4 temporal structure**:

$$H_{ijij}(0,\omega;T) \;=\; \int\!\frac{d^3k_1}{(2\pi)^3}\,\frac{14}{3}\,\frac{1}{2\pi}(W_\Phi*W_\Phi)(k_1,k_1;\omega,T)
\;=\;\frac{7}{3\pi^2}\!\int_0^\infty\!\!dk_1\,k_1^2\,W_{\Phi\Phi}(k_1;\omega,T),$$

after $d^3k_1\to 4\pi k_1^2 dk_1$ (the integrand is angle-independent once $u=k_1$).

With $u=k_1$ both legs share the same $\tau_1$ and $\Phi$, and the §4 convolution simplifies to the single-variable form:

$$W_{\Phi\Phi}(k_1;\omega,T) \;=\; \frac{\Phi^2(k_1,T)\,\tau_1(k_1,T)}{2\pi}\;\mathcal{G}_{\rm SS}\!\bigl(q;\,\xi\bigr),
\qquad
q=\omega\,\tau_1(k_1,T),\quad \xi=\tau_1^2(k_1,T)\,\partial_T^2\ln\Phi(k_1,T),$$

with the **self-similar temporal kernel**

$$\boxed{\;\mathcal{G}_{\rm SS}(q;\xi) \;=\; G_{\rm BK}(q) \;-\; \tfrac{\xi}{4}\,G_{\rm BK}''(q).\;}$$

Here $G_{\rm BK}=\hat g*\hat g$ is the BK2016 convolution kernel of the baseline notebook, and $-G_{\rm BK}''(q) = (\hat g*\hat g_2)(q)$ is the $O(\epsilon^2)$ memory correction. The dimensionless parameter $\xi(k_1,T)$ measures the slow second derivative of $\ln\Phi$ in units of the local eddy time — by §3.2 it is $O(\epsilon^2)$, small in the inertial range.

Using $\Phi(k,T)=E(k,T)/(4\pi k^2)$, so $k_1^2\Phi^2=E^2/(16\pi^2 k_1^2)$, the K2008-reduced source at slow time $T$ becomes

$$\boxed{\;H_{ijij}(0,\omega;T) \;=\; \frac{7}{96\,\pi^5}\!\int_0^\infty\!\!dk_1\;\frac{E^2(k_1,T)}{k_1^2}\;\tau_1(k_1,T)\;\mathcal{G}_{\rm SS}\!\bigl(\omega\tau_1(k_1,T);\,\xi(k_1,T)\bigr).\;}$$

This is the instantaneous (at slow time $T$) GW production kernel in the aeroacoustic limit. The total GW source is the $T$-integral over the emission window (§7).


In [10]:
# H_ijij(0, omega; T):  K2008-reduced GW production at slow time T.
#
#   H(0, omega; T) = 7/(96 pi^5) * int dk k^{-2} E^2(k,T) tau_1(k,T) * G_SS(omega tau_1; xi)

def d2lnPhi_dT2(k, T, dT=None, **pars):
    """Numerical second derivative of ln Phi(k, T) in T, used to compute xi."""
    if dT is None:
        dT = 0.02 * (pars["tau_st"] + T)  # scale step to slow timescale
    f0 = np.log(Phi_eq(k, T,         **pars))
    fp = np.log(Phi_eq(k, T + dT,    **pars))
    fm = np.log(Phi_eq(k, max(T-dT, -0.9*pars["tau_st"]), **pars))
    return (fp - 2*f0 + fm) / dT**2

def xi_param(k, T, **pars):
    """xi(k, T) = tau_1^2(k, T) * d^2 ln Phi(k, T) / dT^2.  Should be O(epsilon^2)."""
    return tau1(k, T, **pars)**2 * d2lnPhi_dT2(k, T, **pars)

def H_ss_T(omega, T, k_min=1e-2, k_max=1e3, n_k=200, include_xi=True, **pars):
    """Instantaneous self-similar H_ijij(0, omega; T) at slow time T.

    `include_xi=False` drops the O(epsilon^2) correction, recovering the leading-order
    instantaneous BK2016 result evaluated at parameters of slow time T.
    """
    ks = np.geomspace(k_min, k_max, n_k)
    E  = np.array([E_kt(k, T, **pars) for k in ks])
    t1 = np.array([tau1(k, T, **pars) for k in ks])
    q  = omega * t1
    if include_xi:
        xi = np.array([xi_param(k, T, **pars) for k in ks])
    else:
        xi = np.zeros_like(ks)
    Gss = G_SS(q, xi)
    integrand = (E**2 / ks**2) * t1 * Gss
    return 7.0 / (96.0 * np.pi**5) * np.trapezoid(integrand, ks)


## 6. Self-similar coordinates and dimensional counting

Switch the wavenumber integration in §5 to the self-similar variable

$$\kappa \;\equiv\; k\,L(T),
\qquad dk = d\kappa/L(T),
\qquad E(k,T) = u^2(T)\,L(T)\,\phi(\kappa).$$

The eddy time and dimensionless frequency become

$$\tau_1(k,T) \;=\; \frac{\tau_e(T)}{C_\varepsilon^{1/3}}\,\kappa^{-2/3},
\qquad \omega\,\tau_1(k,T) \;=\; \frac{\Omega(T)}{C_\varepsilon^{1/3}}\,\kappa^{-2/3},
\qquad \tau_e(T)\equiv L(T)/u(T),\;\;\Omega(T)\equiv\omega\,\tau_e(T),$$

with $\varepsilon(T)=C_\varepsilon u^3(T)/L(T)$ from §V.a. Substituting into the $k$-integrand,

$$\frac{E^2(k,T)}{k^2}\,\tau_1(k,T)\,dk
\;=\; \frac{u^3(T)\,L^4(T)}{C_\varepsilon^{1/3}}\;\frac{\phi^2(\kappa)}{\kappa^{8/3}}\,d\kappa,$$

so the §5 result rewrites as

$$\boxed{\;H_{ijij}(0,\omega;T) \;=\; \frac{7\,u^3(T)\,L^4(T)}{96\,\pi^5\,C_\varepsilon^{1/3}}\;\mathcal{J}\!\bigl(\Omega(T)\bigr),\;}$$

with the **universal dimensionless integral**

$$\mathcal{J}(\Omega) \;\equiv\; \int_0^\infty\!d\kappa\;\frac{\phi^2(\kappa)}{\kappa^{8/3}}\;\mathcal{G}_{\rm SS}\!\Bigl(\Omega\,C_\varepsilon^{-1/3}\kappa^{-2/3};\;\xi(\kappa)\Bigr).$$

### 6.1 $\xi$ is a universal function of $\kappa$ (Loitsiansky closure)

The $O(\epsilon^2)$ parameter $\xi(k,T)=\tau_1^2(k,T)\,\partial_T^2\ln\Phi(k,T)$ from §3.4 looks $T$-dependent, but the Loitsiansky closure $q+p/2=1$ (§V.a) collapses the $T$-dependence into an overall constant.

Direct computation: with $\partial_T\ln L=q/(\tau_\ast+T)$, $\partial_T\ln u=-p/[2(\tau_\ast+T)]$, and $\partial_T\ln\phi(\kappa)=\kappa(\ln\phi)'(\kappa)\,\partial_T\ln L$,

$$\partial_T\ln\Phi(k,T)
\;=\; \frac{G_\phi(\kappa)}{\tau_\ast+T},\qquad
G_\phi(\kappa)\equiv -p+q+q\,\kappa\,(\ln\phi)'(\kappa),$$

$$\partial_T^2\ln\Phi(k,T)
\;=\; \frac{H_\phi(\kappa)}{(\tau_\ast+T)^2},\qquad
H_\phi(\kappa)\equiv q\,\kappa\,G_\phi'(\kappa)-G_\phi(\kappa).$$

Since $\tau_1(k,T)=\tau_e(T)\,C_\varepsilon^{-1/3}\kappa^{-2/3}$ and the closure forces $\tau_e(T)/(\tau_\ast+T)=\tau_{e,0}/\tau_\ast\equiv\epsilon_0$ (a constant!),

$$\boxed{\;\xi(k,T) \;=\; \epsilon_0^{\,2}\,C_\varepsilon^{-2/3}\,\kappa^{-4/3}\,H_\phi(\kappa)\;\equiv\;\epsilon_0^{\,2}\,\bar\xi(\kappa).\;}$$

$\xi$ depends **only** on $\kappa$ (and on the chosen $\phi$), times an overall factor $\epsilon_0^2 = (\tau_{e,0}/\tau_\ast)^2$ that measures the strength of self-similar decay. The universal $\mathcal{J}(\Omega)$ is a function of $\Omega$ alone (and parametrically $\epsilon_0$).

### 6.2 Time dependence of the prefactor

For shifted-time Loitsiansky power laws,
$$u^3(T)\,L^4(T) \;=\; u_0^3\,L_0^4\,(1+T/\tau_\ast)^{-3p/2+4q}.$$
For Loitsiansky $(p,q)=(10/7,2/7)$, the exponent is $-15/7+8/7 = -1$ exactly:
$$u^3 L^4 \;=\; \frac{u_0^3\,L_0^4}{1+T/\tau_\ast}.$$
(For Saffman $(p,q)=(6/5,2/5)$ the exponent is $-1/5$.) The exact $-1$ for Loitsiansky is what makes the $T$-integral in §7 logarithmic.


## 7. Main result

Integrating the slow-time source from §6 over the emission window $T\in[0,T_{\rm em}]$:

$$H_{ijij}(0,\omega) \;=\; \int_0^{T_{\rm em}}\!dT\;\frac{7\,u^3(T)\,L^4(T)}{96\,\pi^5\,C_\varepsilon^{1/3}}\;\mathcal{J}\!\bigl(\Omega(T)\bigr).$$

**Loitsiansky case.** Using $u^3 L^4 = u_0^3 L_0^4/(1+T/\tau_\ast)$ from §6.2, and $\Omega(T)=\omega\tau_e(T) = \omega\tau_{e,0}(1+T/\tau_\ast)$ from the closure of §V.a, change variable to $\Omega$:

$$dT \;=\; \frac{\tau_\ast\,d\eta}{1},\qquad \eta=1+T/\tau_\ast,\qquad \Omega=\omega\tau_{e,0}\eta,\qquad \frac{d\Omega}{\Omega}=\frac{d\eta}{\eta}=\frac{dT}{\tau_\ast+T}.$$

The integrand $u^3 L^4\,dT$ becomes $u_0^3 L_0^4\,\tau_\ast\,d\Omega/\Omega$, giving the closed form

$$\boxed{\;H_{ijij}(0,\omega) \;=\; \frac{7\,u_0^3\,L_0^4\,\tau_\ast}{96\,\pi^5\,C_\varepsilon^{1/3}}\,\int_{\Omega_0}^{\Omega_{\rm em}}\!\frac{d\Omega}{\Omega}\;\mathcal{J}(\Omega;\,\epsilon_0),\;}$$

with the integration limits

$$\Omega_0 \;=\; \omega\,\tau_{e,0},\qquad \Omega_{\rm em} \;=\; \omega\,\tau_{e,0}(1+T_{\rm em}/\tau_\ast) \;\equiv\; \omega\,\tau_e(T_{\rm em}).$$

### 7.1 Reading of the result

- The decay enters through the **integration range** in dimensionless frequency: the GW source picks up contributions from every $\Omega \in [\Omega_0,\Omega_{\rm em}]$ as the eddy time $\tau_e(T)$ drifts during emission.
- The **logarithmic measure** $d\Omega/\Omega$ is special to the Loitsiansky closure ($-3p/2+4q=-1$). For Saffman the measure becomes $\eta^{-1/5}\,d\eta = \eta^{-6/5}\,d\Omega/(\omega\tau_{e,0})$ — a different power-law tilt that the same code reproduces by switching $p$ and $q$.
- The **memory correction** lives inside $\mathcal{J}(\Omega;\epsilon_0)$ via $\mathcal{G}_{\rm SS}(q;\xi)=G_{\rm BK}(q)-(\xi/4)G_{\rm BK}''(q)$, with $\xi=\epsilon_0^2\,\bar\xi(\kappa)$ a $T$-independent kappa-profile (§6.1).

### 7.2 Limits to keep nearby

- **Quasi-stationary limit** $\tau_\ast\to\infty$ at fixed $T_{\rm em}$: $\Omega_{\rm em}-\Omega_0\to 0$, $\int_{\Omega_0}^{\Omega_{\rm em}} d\Omega/\Omega\to T_{\rm em}/\tau_\ast$, and the integrand stays at $\mathcal{J}(\omega\tau_{e,0})$. Result: $H\to (7\,u_0^3 L_0^4\,T_{\rm em}/96\pi^5 C_\varepsilon^{1/3})\,\mathcal{J}_0(\omega\tau_{e,0})$, the BK2016 baseline times the emission duration. (Sanity check in §8.)
- **Long decay** $T_{\rm em}\gg\tau_\ast$: the upper limit is at $\Omega_{\rm em}\gg\Omega_0$, and the integral is dominated by the high-$\Omega$ tail of $\mathcal{J}(\Omega)$ — the **new** decaying-turbulence prediction, which is qualitatively different from the baseline.


In [11]:
# Total H_ijij(0, omega) = int_0^{T_em} dT * H_ss_T(omega, T).

def H_total(omega, T_em, n_T=41, include_xi=True, **pars):
    """Self-similar GW source integrated over emission time T in [0, T_em]."""
    Ts = np.linspace(0.0, T_em, n_T)
    vals = np.array([H_ss_T(omega, T, include_xi=include_xi, **pars) for T in Ts])
    return np.trapezoid(vals, Ts)

def H_frozen(omega, T_em, **pars):
    """Frozen-source baseline: evaluate the instantaneous source at T=0 (with eps_0, u_0, L_0)
    and multiply by the emission duration T_em. This is the BK2016 baseline, not a
    tau_* -> infinity limit (which would drive eps -> 0 unphysically)."""
    return T_em * H_ss_T(omega, 0.0, include_xi=False, **pars)


## 8. Quasi-stationary limit — sanity check

Sending $\tau_\ast\to\infty$ at fixed $T_{\rm em}$ freezes the source: $u(T)\to u_0$, $L(T)\to L_0$, $\varepsilon(T)\to\varepsilon_0$, and the $W_\Phi^{(1)}$ correction goes to zero (because $\partial_T^2\ln\Phi\to 0$). The boxed §7 result then reduces to

$$H_{ijij}(0,\omega) \xrightarrow{\;\tau_\ast\to\infty\;}\; \frac{7\,u_0^3 L_0^4\,T_{\rm em}}{96\,\pi^5\,C_\varepsilon^{1/3}}\;\mathcal{J}_0(\omega\tau_{e,0}),$$

which is the BK2016 baseline (instantaneous source × emission duration) — the result of `decaying_turbulence_gw_spectrum_analytic.ipynb` on `main`. **This is the non-negotiable consistency check.** The cell below evaluates both expressions on the same grid.


In [12]:
# Sanity: at large tau_*, H_total -> H_frozen.

pars_chk = {"u0": 1.0, "l0": 1.0, "tau_st": 1.0, "p": 10/7, "q": 2/7, "s": 4}
omegas_chk = np.geomspace(0.3, 5.0, 6)

# Three values of tau_*: short decay, comparable decay, long (quasi-stationary)
for tau_st_val, label in [(0.5, "fast decay"), (5.0, "moderate"), (1e4, "quasi-stationary")]:
    p = {**pars_chk, "tau_st": tau_st_val}
    H_dyn = np.array([H_total(om, T_em=1.0, include_xi=False, **p) for om in omegas_chk])
    H_fr  = np.array([H_frozen(om, T_em=1.0, **p)                 for om in omegas_chk])
    ratio = H_dyn / H_fr
    print(f"tau_* = {tau_st_val:.1e}  ({label})")
    print(f"  omega:    " + "  ".join(f"{o:7.3f}" for o in omegas_chk))
    print(f"  H_dyn/H_frozen: " + "  ".join(f"{r:7.4f}" for r in ratio))
    print()
print("For tau_* -> infinity the ratio should approach 1.")


tau_* = 5.0e-01  (fast decay)
  omega:      0.300    0.527    0.924    1.623    2.848    5.000
  H_dyn/H_frozen:  0.3751   0.3635   0.3373   0.3078   0.2813   0.2599

tau_* = 5.0e+00  (moderate)
  omega:      0.300    0.527    0.924    1.623    2.848    5.000
  H_dyn/H_frozen:  0.8495   0.8369   0.8218   0.8064   0.7916   0.7812

tau_* = 1.0e+04  (quasi-stationary)
  omega:      0.300    0.527    0.924    1.623    2.848    5.000
  H_dyn/H_frozen:  0.9999   0.9999   0.9997   1.0003   1.0010   1.0000

For tau_* -> infinity the ratio should approach 1.


## 9. Connection to code

Current `src/gw_turbulence/core.py` exposes (all quasi-stationary):

- `g_decaying(z)` -- dimensionless $\hat{g}(q)$ kernel.
- `_temporal_conv_decay(q)` -- convolution $G_{\rm BK}(q)$.
- `H_pq_decaying(p,q,...)` -- full Gogoberidze grid for BK2016.
- `H_delta_k_decay`, `H_white_decay`, and their `_grid` variants -- toy spatial models with BK2016 temporal.

**TODO -- decide the shape of the new API once Sec. 7 is in place.** Options, roughly in increasing order of invasiveness:

- Add `_temporal_conv_selfsimilar(omega, t, ...)` alongside the existing convolution, no changes to grids.
- Add a new `H_pq_selfsimilar(p, q, M, tau_star, ...)` that takes the decay-time parameter and drops into the same grid machinery.
- Rename the existing `*_decaying` symbols to `*_quasistationary` project-wide to avoid misinterpretation. **Deferred** -- do this only once the new API exists and we know what to name it against.


## 10. Numerical cross-check

Once the self-similar kernel is coded, produce a plot overlaying:

- `H_pq_decaying` (baseline, this branch's quasi-stationary reference).
- The new self-similar kernel at several $\omega\tau_*$.
- The $\omega\tau_*\to\infty$ limit of the self-similar kernel -- should sit on top of the baseline curve.

Use the existing plotting conventions from the baseline notebook (`HD_CLASSES`, log-log axes, shared $q$-grid) so the comparison is apples-to-apples.


In [13]:
pars = {"u0": 1.0, "l0": 1.0, "tau_st": 1.0, "p": 10/7, "q": 2/7, "s": 4}
omegas = np.geomspace(0.1, 10, 16)

H_dyn  = np.array([H_total(om, T_em=10.0, include_xi=False, **pars) for om in omegas])  # leading order
H_full = np.array([H_total(om, T_em=10.0, include_xi=True,  **pars) for om in omegas])  # + O(eps^2)
H_fr   = np.array([H_frozen(om, T_em=10.0, **pars)                  for om in omegas])

fig, ax = plt.subplots()
ax.loglog(omegas, np.abs(H_dyn),  label="self-similar (leading)")
ax.loglog(omegas, np.abs(H_full), label=r"self-similar (+ $O(\epsilon^2)$)", ls=":")
ax.loglog(omegas, np.abs(H_fr),   label="frozen baseline", ls="--")
ax.set_xlabel(r"$\omega$")
ax.set_ylabel(r"$|H_{ijij}(0,\omega)|$")
ax.legend()
plt.tight_layout()
plt.show()


/tmp/ipykernel_42893/3040326608.py:16: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


---

## Scratch

Working notes, intermediate algebra, discarded paths. Keep everything here rather than in the numbered sections until a result is ready to promote.


In [14]:
# scratch cell
